# Gold Price Data Exploration

This notebook demonstrates how to load, explore, and visualize gold price data along with economic indicators.

**Requirements covered:** 1.1-2.6

## Objectives
1. Load gold price OHLCV data
2. Validate data quality
3. Explore data distributions and patterns
4. Analyze correlations with economic indicators
5. Identify missing values and outliers

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from src.data_ingestion import DataIngestionManager
from src.data_preprocessing import DataPreprocessor
from config import Config

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load Gold Price Data

In [ ]:
# Initialize data manager
data_manager = DataIngestionManager()

# Load gold price data
csv_path = '../XAU_1d_data.csv'
gold_df = data_manager.load_csv(csv_path)

print(f"Loaded {len(gold_df)} records")
print(f"Date range: {gold_df['Date'].min()} to {gold_df['Date'].max()}")
print(f"\nColumns: {list(gold_df.columns)}")
gold_df.head()

## 2. Data Validation

In [ ]:
# Validate OHLCV data
validation_result = data_manager.validate_ohlcv_data(gold_df)

print(f"Validation Status: {'✓ PASSED' if validation_result.is_valid else '✗ FAILED'}")
print(f"\nErrors: {len(validation_result.errors)}")
for error in validation_result.errors[:5]:
    print(f"  - {error}")

print(f"\nWarnings: {len(validation_result.warnings)}")
for warning in validation_result.warnings[:5]:
    print(f"  - {warning}")

In [ ]:
# Check for duplicates and chronological order
duplicates = data_manager.check_duplicates(gold_df)
is_chronological = data_manager.validate_chronological_order(gold_df)

print(f"Duplicate dates: {len(duplicates)}")
print(f"Chronologically ordered: {is_chronological}")

## 3. Basic Statistics and Distribution

In [ ]:
# Summary statistics
print("Summary Statistics:")
gold_df[['Open', 'High', 'Low', 'Close', 'Volume']].describe()

In [ ]:
# Plot price distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Price histogram
axes[0, 0].hist(gold_df['Close'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Gold Close Price Distribution')
axes[0, 0].set_xlabel('Price (USD)')
axes[0, 0].set_ylabel('Frequency')

# Volume histogram
axes[0, 1].hist(gold_df['Volume'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Trading Volume Distribution')
axes[0, 1].set_xlabel('Volume')
axes[0, 1].set_ylabel('Frequency')

# Daily returns
gold_df['Returns'] = gold_df['Close'].pct_change()
axes[1, 0].hist(gold_df['Returns'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('Daily Returns Distribution')
axes[1, 0].set_xlabel('Returns')
axes[1, 0].set_ylabel('Frequency')

# Box plot
gold_df[['Open', 'High', 'Low', 'Close']].boxplot(ax=axes[1, 1])
axes[1, 1].set_title('OHLC Price Box Plot')
axes[1, 1].set_ylabel('Price (USD)')

plt.tight_layout()
plt.show()

## 4. Time Series Visualization

In [ ]:
# Plot gold price over time
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Close price
axes[0].plot(gold_df['Date'], gold_df['Close'], linewidth=1, color='gold')
axes[0].set_title('Gold Price Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Close Price (USD)')
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(gold_df['Date'], gold_df['Volume'], width=1, alpha=0.6, color='steelblue')
axes[1].set_title('Trading Volume Over Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Load Economic Indicators

In [ ]:
# Load economic indicators
start_date = gold_df['Date'].min().strftime('%Y-%m-%d')
end_date = gold_df['Date'].max().strftime('%Y-%m-%d')

print(f"Fetching economic indicators from {start_date} to {end_date}...")
tickers = list(Config.INDICATORS.values())

try:
    indicators = data_manager.load_economic_indicators(tickers, start_date, end_date)
    print(f"\nLoaded indicators:")
    for name, df in indicators.items():
        print(f"  - {name}: {len(df)} records")
except Exception as e:
    print(f"Error loading indicators: {e}")
    print("Continuing with gold data only...")
    indicators = {}

## 6. Correlation Analysis with Economic Indicators

In [ ]:
if indicators:
    # Align datasets
    preprocessor = DataPreprocessor()
    combined_df = preprocessor.align_datasets(gold_df, indicators)
    
    print(f"Combined dataset: {len(combined_df)} records")
    print(f"Columns: {list(combined_df.columns)}")
    
    # Calculate correlations
    numeric_cols = combined_df.select_dtypes(include=[np.number]).columns
    correlation_matrix = combined_df[numeric_cols].corr()
    
    # Plot correlation heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix: Gold Prices and Economic Indicators', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print key correlations with gold close price
    print("\nCorrelations with Gold Close Price:")
    close_corr = correlation_matrix['Close'].sort_values(ascending=False)
    for col, corr in close_corr.items():
        if col != 'Close':
            print(f"  {col}: {corr:.3f}")
else:
    print("No economic indicators loaded. Skipping correlation analysis.")

In [ ]:
if indicators and 'DXY' in combined_df.columns:
    # Plot gold vs DXY (US Dollar Index)
    fig, ax1 = plt.subplots(figsize=(15, 6))
    
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Gold Price (USD)', color='gold')
    ax1.plot(combined_df['Date'], combined_df['Close'], color='gold', linewidth=2, label='Gold')
    ax1.tick_params(axis='y', labelcolor='gold')
    ax1.grid(True, alpha=0.3)
    
    ax2 = ax1.twinx()
    ax2.set_ylabel('DXY (US Dollar Index)', color='steelblue')
    ax2.plot(combined_df['Date'], combined_df['DXY'], color='steelblue', linewidth=2, label='DXY')
    ax2.tick_params(axis='y', labelcolor='steelblue')
    
    plt.title('Gold Price vs US Dollar Index (DXY)', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()

## 7. Data Quality Assessment

In [ ]:
# Check missing values
missing_stats = gold_df.isnull().sum()
missing_pct = (missing_stats / len(gold_df)) * 100

print("Missing Values:")
for col, count in missing_stats.items():
    if count > 0:
        print(f"  {col}: {count} ({missing_pct[col]:.2f}%)")

if missing_stats.sum() == 0:
    print("  No missing values found! ✓")

In [ ]:
# Detect outliers using z-score method
from scipy import stats

z_scores = np.abs(stats.zscore(gold_df[['Close', 'Volume']].dropna()))
outliers = (z_scores > 3).any(axis=1)
outlier_count = outliers.sum()

print(f"Outliers detected (>3 std): {outlier_count} records ({outlier_count/len(gold_df)*100:.2f}%)")

if outlier_count > 0:
    outlier_df = gold_df[['Date', 'Close', 'Volume']][outliers]
    print("\nSample outliers:")
    print(outlier_df.head())

## 8. Generate Data Quality Report

In [ ]:
# Initialize preprocessor and generate quality report
preprocessor = DataPreprocessor()

# Preprocess data to generate report
processed_df = preprocessor.handle_missing_values(gold_df.copy())
processed_df = preprocessor.remove_outliers(processed_df)

quality_report = preprocessor.generate_quality_report()

print("="*60)
print("DATA QUALITY REPORT")
print("="*60)
print(f"Total Records: {quality_report.total_records}")
print(f"Date Range: {quality_report.date_range[0]} to {quality_report.date_range[1]}")
print(f"Quality Score: {quality_report.data_quality_score:.1f}/100")
print(f"\nMissing Values Handled:")
for col, count in quality_report.missing_values_handled.items():
    print(f"  {col}: {count}")
print(f"\nOutliers Removed:")
for col, count in quality_report.outliers_removed.items():
    print(f"  {col}: {count}")
print("="*60)

## Summary

This notebook demonstrated:
1. ✅ Loading and validating gold price OHLCV data
2. ✅ Exploring data distributions and statistics
3. ✅ Visualizing time series trends
4. ✅ Loading and correlating economic indicators (DXY, Oil, Treasury yields)
5. ✅ Assessing data quality (missing values, outliers)
6. ✅ Generating comprehensive quality reports

**Next Steps:**
- Proceed to `02_model_training.ipynb` for model training
- Or explore `03_prediction_and_forecasting.ipynb` for predictions